# IMDB Binary Text Classification with Neural Networks

## Project Overview

In this mini-project, we build a binary sentiment classifier using the IMDB movie reviews dataset.

Objectives:

- Preprocess text data using one-hot encoding.
- Build a feedforward neural network.
- Train and validate the model.
- Detect overfitting using learning curves.
- Retrain using an optimal number of epochs.
- Evaluate performance on the test set.


## 1. Load the Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import models, layers

num_words = 10_000

(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(
    num_words=num_words
)

print("Training samples:", len(train_data))
print("Test samples:", len(test_data))


The IMDB dataset contains movie reviews represented as sequences of integers. Each integer corresponds to a word index in the vocabulary.


## 2. Preprocess the Data

In [ ]:
def vectorize_sequences(sequences, dimension=10000):

    results = np.zeros((len(sequences), dimension))

    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1

    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

print(x_train.shape)
print(x_test.shape)


The reviews are transformed into binary vectors of length 10,000. Each position indicates whether a word appears in the review.


## 3. Create Validation Set

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]

y_val = y_train[:10000]
partial_y_train = y_train[10000:]

print("Training set:", partial_x_train.shape)
print("Validation set:", x_val.shape)


We reserve the first 10,000 samples for validation and use the remaining samples for training.


## 4. Build the Neural Network

### Architecture

The network contains:

- Two hidden Dense layers.
- ReLU activation functions.
- One sigmoid output neuron.

Because this is a binary classification problem, we use binary crossentropy as the loss function.


In [ ]:
model = models.Sequential([

    layers.Dense(16, activation='relu', input_shape=(10000,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')

])

model.summary()


## 5. Compile the Model

In [ ]:
model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


### Why RMSprop?

RMSprop adapts learning rates during training and performs well on many text classification tasks.


## 6. Train the Model

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val)
)


## 7. Visualize Training History

In [ ]:
history_dict = history.history

loss_values = history_dict['loss']
val_loss_values = history_dict['val_loss']

epochs = range(1, len(loss_values)+1)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(epochs, loss_values, label='Training Loss')
plt.plot(epochs, val_loss_values, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, history_dict['accuracy'], label='Training Accuracy')
plt.plot(epochs, history_dict['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.show()


### Overfitting Analysis

Observe where validation loss starts increasing while training loss continues decreasing.

That point indicates the beginning of overfitting.

Typically for IMDB, overfitting begins around epoch 3–5.


## 8. Retrain Using Optimal Number of Epochs

In [ ]:
optimal_epochs = 4

final_model = models.Sequential([

    layers.Dense(16, activation='relu', input_shape=(10000,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')

])

final_model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

final_model.fit(
    x_train,
    y_train,
    epochs=optimal_epochs,
    batch_size=512
)


The optimal number of epochs should be selected based on the learning curves generated previously.


## 9. Evaluate on Test Data

In [ ]:
results = final_model.evaluate(
    x_test,
    y_test
)

print("Test Loss :", results[0])
print("Test Accuracy :", results[1])


## 10. Results Analysis

### Discussion

Compare:

- Training accuracy
- Validation accuracy
- Validation loss

If training performance continues improving while validation performance degrades, the model is overfitting.

Using an optimal number of epochs improves generalization.


## 11. Save the Model

In [ ]:
final_model.save("imdb_sentiment_model.keras")


Saving the trained model allows future inference without retraining.


## Conclusion

In this project we:

- Loaded the IMDB dataset.
- Vectorized text using one-hot encoding.
- Built a feedforward neural network.
- Trained and validated the model.
- Detected overfitting through learning curves.
- Retrained using an optimal number of epochs.
- Evaluated final performance on the test set.

This workflow is a foundation for more advanced NLP tasks such as sentiment analysis, text classification, and sequence modeling.
